# 05. Runnable 기본 조합 (LCEL)

- LangChain에서의 Runnable은 입력값을 받아 특정 작업을 수행하고 출력값을 반환하는 실행 가능한 구성요소임.
- 이를 활용하면 프롬프트, 모델, 출력 파서, 사용자 정의 함수, 병렬 처리 흐름을 직관적인 파이프라인으로 표현할 수 있음.
- LCEL(LangChain Expression Language)은 `|` 연산자로 컴포넌트를 체인처럼 연결함.
- Runnable 조합을 사람이 읽기 쉬운 파이프라인 문법으로 표현한 것.


```
prompt | llm | parser
```

In [2]:
# from dotenv import load_dotenv
# load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [3]:
import sys
sys.path.append("..") # 상위 폴더 path설정
from common_config import llm_connect
model = "gpt-5.4-mini"
llm = llm_connect(model=model)

## 1. 기본 체인: prompt | llm | parser

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}에 대해 한 문장으로 설명해줘")
])

parser = StrOutputParser()

# | 연산자로 체인 구성
chain = prompt | llm | parser # prompt -> llm -> parser

result = chain.invoke({"topic": "벡터 데이터베이스"})
print(result)

벡터 데이터베이스는 **텍스트, 이미지 같은 데이터를 의미를 담은 숫자 벡터로 저장하고, 비슷한 것끼리 빠르게 찾도록 설계된 데이터베이스**입니다.


- 벡터 : 1차원 숫자 데이터

## 2. 체인 스트리밍
- LangChain 체인 실행 결과를 스트리밍 방식으로 조금씩 받아 출력하는 코드

In [5]:
for chunk in chain.stream({"topic": "임베딩"}):
    # 현재 받은 응답 조각을 출력 후 줄바꿈 없이, 버퍼에 쌓아두지 않고 즉시 출력
    print(chunk, end="", flush=True)

임베딩은 **단어나 문장, 이미지 같은 데이터를 컴퓨터가 이해하고 비교할 수 있도록 숫자 벡터로 바꾸는 방법**입니다.

In [6]:
for chunk in chain.stream({"topic": "임베딩"}):
    # Python이 출력 내용을 잠시 버퍼에 모
    # 아두었다가 나중에 보여줌
    print(chunk, end="", flush=False)

임베딩은 **텍스트나 이미지 같은 데이터를 컴퓨터가 의미를 이해할 수 있도록 숫자 벡터로 바꾸는 표현 방식**입니다.

# 3. Runnable의 공통 실행 메서드
- 입력 값을 받아 어떤 작업을 수행하고, 출력값을 반환하는 표준 실행 단위
- 프롬프트, LLM, 출력파서, Retriever, 사용자 정의 함수 등을 하나의 실행 규칙으로 연결하기 위한 표준 인터페이스
- 기본 : 입력 → Runnable → 출력
- 여러 Runnable 연결 : 입력 → prompt → llm → parser → custom function → 최종 출력 

- Runnable의 공통 실행 메서드 

| 메서드                    | 의미              | 사용 상황           |
| ---------------------- | --------------- | --------------- |
| `invoke()`             | 단일 입력 실행        | 가장 기본 실행        |
| `ainvoke()`            | 비동기 단일 실행       | async 환경        |
| `batch()`              | 여러 입력을 한 번에 실행  | 여러 질문 일괄 처리     |
| `abatch()`             | 비동기 batch 실행    | 대량 처리           |
| `stream()`             | 결과를 조각 단위로 스트리밍 | 챗봇 실시간 출력       |
| `astream()`            | 비동기 스트리밍        | 웹소켓, SSE 등      |
| `batch_as_completed()` | 완료되는 순서대로 결과 반환 | 입력별 처리 시간이 다를 때 |


## 3. RunnablePassthrough - 입력을 그대로 전달
- RunnablePassthrough는 입력값을 거의 그대로 통과시킴

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# 입력값을 그대로 흘려보내기
chain = RunnablePassthrough()
result = chain.invoke({"topic": "RAG"})
print(result)

{'topic': 'RAG'}


- `.assign()`은 기존 딕셔너리에 새로운 필드를 추가할 때 사용함

In [10]:
# 실용 예: 입력을 그대로 유지하면서 추가 키 삽입
from langchain_core.runnables import RunnablePassthrough

upper=lambda x: x["word"].upper()
chain = RunnablePassthrough.assign(upper=upper)

result = chain.invoke({"word": "hello"})
print(result)  # {'word': 'hello', 'upper': 'HELLO'}

{'word': 'hello', 'upper': 'HELLO'}


## 4. RunnableLambda - 함수를 Runnable로 감싸기
- RunnableLambda는 일반 파이썬 함수를 Runnable로 감싸는 객체
- RunnableLambda는 체인 중간이나 마지막에 후처리 로직을 넣을 때 주로 사용

In [13]:
from langchain_core.runnables import RunnableLambda

# 문자열 -> 문자열 편집 반환 함수
def add_prefix(text: str) -> str:
    return f"[결과] {text}"

prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}이 뭔가요?")
])

chain = prompt | llm | StrOutputParser() | RunnableLambda(add_prefix)

result = chain.invoke({"topic": "LangGraph"})
print(result)

[결과] LangGraph는 자연어 처리(NLP)와 관련된 기술이나 도구로, 언어와 그래프 이론을 결합하여 언어 데이터를 분석하고 처리하는 방법을 제공하는 시스템이나 프레임워크일 수 있습니다. 일반적으로 이러한 시스템은 언어의 구조적 특성을 그래프 형태로 모델링하여, 단어, 문장, 또는 문서 간의 관계를 시각화하고 분석하는 데 사용됩니다.

LangGraph의 구체적인 기능이나 목적은 사용되는 맥락에 따라 다를 수 있으며, 예를 들어, 언어 모델링, 정보 검색, 텍스트 요약, 또는 의미적 관계 분석 등에 활용될 수 있습니다. 

더 구체적인 정보가 필요하시다면, LangGraph의 특정 구현이나 사용 사례에 대해 말씀해 주시면 추가적인 설명을 드릴 수 있습니다.


## 5. 병렬 실행 - RunnableParallel

딕셔너리 형태로 여러 체인을 동시에 실행합니다.

In [7]:
from langchain_core.runnables import RunnableParallel

prompt_ko = ChatPromptTemplate.from_messages([("human", "{topic}을 한국어로 설명해줘")])
prompt_en = ChatPromptTemplate.from_messages([("human", "Explain {topic} in English")])

chain_ko = prompt_ko | llm | StrOutputParser()
chain_en = prompt_en | llm | StrOutputParser()

# 두 체인을 병렬로 실행하여 각각 한국어와 영어 설명 생성
parallel_chain = RunnableParallel(
    korean=chain_ko,
    english=chain_en
)

result = parallel_chain.invoke({"topic": "RAG"})
print("[한국어]", result["korean"])
print("[English]", result["english"])

[한국어] RAG는 **Retrieval-Augmented Generation**의 약자로, 한국어로는 보통 **검색 증강 생성**이라고 합니다.

쉽게 말하면:

- **생성형 AI**가 답변을 만들기 전에
- **외부 문서나 데이터베이스에서 관련 정보를 먼저 검색(retrieval)** 하고
- 그 검색 결과를 바탕으로 **더 정확한 답변을 생성(generation)** 하는 방식입니다.

## 왜 필요한가?
일반적인 AI는 학습된 지식만으로 답하기 때문에:
- 최신 정보에 약할 수 있고
- 회사 내부 문서 같은 비공개 자료는 모를 수 있으며
- 근거 없이 그럴듯한 답을 만들기도 합니다.

RAG는 이런 문제를 줄여줍니다.

## 동작 방식
1. 사용자가 질문을 입력합니다.
2. 시스템이 관련 문서, DB, 웹자료 등을 검색합니다.
3. 찾은 내용을 AI 모델에 함께 전달합니다.
4. AI가 그 자료를 참고해서 답변을 생성합니다.

## 장점
- **정확도 향상**
- **최신 정보 반영 가능**
- **사내 문서/매뉴얼 활용 가능**
- **환각(hallucination) 감소**

## 예시
예를 들어,  
“우리 회사 연차 규정이 어떻게 돼?”라고 물으면

- 일반 AI: 학습된 일반 지식으로만 대답할 수 있음
- RAG: 회사 인사 규정 문서를 검색한 뒤, 그 내용을 바탕으로 답변

원하시면 제가 **RAG를 그림처럼 구조적으로 설명**해드리거나, **LLM과 RAG의 차이**도 한국어로 쉽게 설명해드릴게요.
[English] RAG stands for **Retrieval-Augmented Generation**.

In simple English, it means:

1. **Retrieve**: first, the system looks up relevant information from a source, like documents, a database, or the web.
2. **Augment**: it uses that retrieved

```mermaid
flowchart LR
    A["📥 입력"] --> B{"⚡ RunnableParallel"}

    B --> C["작업 A<br/>Retriever"]
    B --> D["작업 B<br/>RunnablePassthrough"]
    B --> E["작업 C<br/>추가 처리"]

    C --> F["context: 검색 결과"]
    D --> G["question: 원본 입력"]
    E --> H["result: 처리 결과"]

    F --> I["📦 하나의 객체로 병합"]
    G --> I
    H --> I

    I --> J["📤 다음 Runnable로 전달"]

    style A fill:#E3F2FD,stroke:#1976D2,stroke-width:2px
    style B fill:#FFF3E0,stroke:#F57C00,stroke-width:3px
    style C fill:#E8F5E9,stroke:#388E3C
    style D fill:#F3E5F5,stroke:#7B1FA2
    style E fill:#E0F7FA,stroke:#0097A7
    style I fill:#FFFDE7,stroke:#F9A825,stroke-width:2px
    style J fill:#E8EAF6,stroke:#3F51B5,stroke-width:2px
```

## 실습 문제
Q2. RunnableLambda를 활용해 LLM 응답을 받아 단어 수를 세는 함수를 파이프라인 끝에 추가해보세요. 결과로 {'text': '...', 'word_count': 42} 형태로 반환하세요.

In [ ]:
from langchain_core.runnables import RunnableLambda

# 1. 프롬프트 구성
prompt = ChatPromptTemplate.from_template(
    "{topic}에 대해 초보자도 이해할 수 있게 설명해줘."
)

# 2. LLM 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# 3. 출력 파서 생성
parser = StrOutputParser()

# 4. LLM 응답을 받아 단어 수를 세는 함수
def count_words(text: str) -> dict:
    return {
        "text": text,
        "word_count": len(text.split())
    }

# 5. RunnableLambda로 함수 감싸기
word_count_runnable = RunnableLambda(count_words)

# 6. LCEL 체인 구성
chain = prompt | llm | parser | word_count_runnable

# 7. 실행
result = chain.invoke({"topic": "LangChain Runnable"})

print(result["word_count"])  # 설명의 단어 수
print("-"*20)
print(result["text"])        # LLM이 생성한 설명

175
--------------------
LangChain은 자연어 처리(NLP)와 관련된 다양한 작업을 수행할 수 있도록 도와주는 프레임워크입니다. 그 중에서도 "Runnable"은 LangChain에서 작업을 수행하는 기본 단위입니다. 초보자도 이해할 수 있도록 쉽게 설명해볼게요.

### Runnable이란?

1. **작업 단위**: Runnable은 특정 작업을 수행하는 코드 블록입니다. 예를 들어, 텍스트를 분석하거나, 질문에 답변하거나, 데이터를 변환하는 등의 작업을 할 수 있습니다.

2. **입력과 출력**: Runnable은 입력을 받아서 처리한 후, 결과를 출력합니다. 예를 들어, 사용자가 "안녕하세요"라는 텍스트를 입력하면, Runnable은 이를 처리하여 "안녕하세요! 어떻게 도와드릴까요?"와 같은 응답을 생성할 수 있습니다.

3. **조합 가능성**: 여러 개의 Runnable을 조합하여 더 복잡한 작업을 수행할 수 있습니다. 예를 들어, 첫 번째 Runnable이 텍스트를 정리하고, 두 번째 Runnable이 그 텍스트를 분석하는 식으로 연결할 수 있습니다.

### 예시

- **텍스트 변환**: 사용자가 입력한 문장을 대문자로 변환하는 Runnable을 만들 수 있습니다.
- **질문 응답**: 사용자가 질문을 입력하면, 그에 대한 답변을 생성하는 Runnable을 만들 수 있습니다.

### 요약

Runnable은 LangChain에서 작업을 수행하는 기본적인 구성 요소로, 입력을 받아서 처리하고 출력을 생성하는 역할을 합니다. 여러 개의 Runnable을 조합하여 복잡한 작업을 수행할 수 있는 유연성을 제공합니다. 초보자도 쉽게 사용할 수 있도록 설계되어 있어, 자연어 처리 작업을 간편하게 수행할 수 있습니다.
